# MiniCells CLM-0.1 Research Preview — Release Build
This notebook reproduces the locked Geometry upcycling candidates, runs Conditionality Validation 002, benchmarks the release candidate, builds the public bundle, and optionally publishes it. Use Internet On and T4 x2 when available.

In [ ]:
import os, pathlib, subprocess, sys
ROOT = pathlib.Path('/kaggle/working/mini-cells')
REF = os.environ.get('MINICELLS_REF', 'release/clm-0.1-prep')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/ArcheLabs/mini-cells.git',str(ROOT)], check=True)
subprocess.run(['git','fetch','origin'], cwd=ROOT, check=True)
subprocess.run(['git','checkout',REF], cwd=ROOT, check=True)
subprocess.run(['git','reset','--hard',f'origin/{REF}'], cwd=ROOT, check=True)
print(subprocess.check_output(['git','log','-1','--oneline'], cwd=ROOT, text=True))
subprocess.run([sys.executable,'-m','pip','install','-e','.[lm,dev]'], cwd=ROOT, check=True)
# PEP 660 editable installs are discovered by fresh Python processes via site .pth files.
# This already-running Kaggle kernel does not reprocess those .pth files, so expose the
# src-layout package explicitly as well. Subprocess workers still use the editable install.
research_path = str(ROOT/'research')
if research_path not in sys.path:
    sys.path.insert(0, research_path)
import minicells
print('minicells kernel import:', pathlib.Path(minicells.__file__).resolve())


In [ ]:
# CPU/static preflight before any CUDA training.
for script in ['scripts/run_clm_0_1_release_worker.py','scripts/run_clm_0_1_release.py','scripts/publish_clm_0_1_release.py']:
    subprocess.run([sys.executable,'-m','py_compile',script], cwd=ROOT, check=True)
subprocess.run([sys.executable,'-m','pytest','tests/test_clm_upcycling_study_001.py','tests/test_clm_0_1_release.py','-q'], cwd=ROOT, check=True)


In [ ]:
# Reproduce 3 Geometry replicas, run Validation 002, benchmark, and build release bundle.
subprocess.run([sys.executable,'scripts/run_clm_0_1_release.py','--fresh'], cwd=ROOT, check=True)


In [ ]:
import json
OUT = ROOT/'results'/'clm-0.1-release'
decision = json.loads((OUT/'decision.json').read_text())
conditionality = json.loads((OUT/'conditionality-002-decision.json').read_text())
benchmark = json.loads((OUT/'benchmark.json').read_text())
print(json.dumps(decision, indent=2))
print(json.dumps(conditionality, indent=2))
print(json.dumps(benchmark, indent=2))
print((OUT/'MODEL_CARD.md').read_text())


In [ ]:
# Public API smoke test from the exact generated bundle.
from minicells import CLM
bundle = OUT/'bundle'/'minicells-clm-0.1'
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
model = CLM.from_pretrained(bundle, device=device)
sample = model.generate('Once upon a time', max_new_tokens=32, temperature=0.8, top_k=40, seed=7, return_routing=True)
print(sample.text)
print('routing steps:', len(sample.routing_usage))
print('first routing usage:', sample.routing_usage[0] if sample.routing_usage else None)


In [ ]:
# Publish only after reviewing all release gates and the generated sample.
PUBLISH = False
if PUBLISH:
    subprocess.run([sys.executable,'scripts/publish_clm_0_1_release.py','--push'], cwd=ROOT, check=True)
else:
    print('PUBLISH=False; release artifacts were not pushed.')
